# 02 商业分析模型

**本笔记本作用**：跑通 2.2 要求的全部商业分析模型（共 7 类），对应论文第 3 章。

| 编号 | 模型 | 类别 | 关键参数与业务依据 |
| --- | --- | --- | --- |
| M1 | 同比 / 环比 + 3 期移动平均 | 趋势与对比 | 同比取隔 12 个月；移动平均窗口 3，抹平单月波动 |
| M2 | 帕累托 / ABC 分类 | 结构与价值 | 70%/90% 为库存 ABC 行业惯例切分 |
| M3 | RFM 八分类 | 用户与客群 | 三项均用**中位数**切分（分布右偏，均值会被大客户拉高） |
| M4 | K-Means 用户分群 | 用户与客群 | log1p + 标准化；K 在 3~6 用轮廓系数择优 |
| M5 | 同期群留存 | 用户与客群 | 以首购月为同期群，群规模 ≥20 才纳入，过滤小样本噪声 |
| M6 | 波士顿矩阵 | 结构与价值 | 取 2025/2026 两个完整年；分界线=平均份额与整体增速 |
| M7 | SARIMA vs Holt-Winters | 预测与预警 | 网格搜 AIC；以**样本外 6 个月 MAPE** 择优；输出 95% 置信区间 |
| M8 | Apriori 关联规则 | 关联与转化 | 订单=购物篮、子类=商品；支持度 0.6%、置信度 15%、lift>1 |
| M9 | 漏斗转化 | 关联与转化 | 下单→按时履约→未退货→盈利 四级漏斗 |

> 要求 ≥3 种，本项目实际完成 **7 类 / 9 个模型**，每个模型都产出结果表与业务结论。

In [1]:
# ============================================================
# 运行环境：conda 环境 business_analysis | 内核 business_analysis
# 本单元格作用：把 scripts 目录加入路径，并提供按文件名加载脚本的工具函数
# ============================================================
import sys, pathlib, importlib.util

ROOT = pathlib.Path.cwd().parent          # 项目根目录（notebooks 的上一级）
sys.path.insert(0, str(ROOT / "scripts"))  # 让脚本里的 import common 生效

def load_step(module_name: str, filename: str):
    """按文件名加载 scripts 下的脚本（文件名以数字开头，不能直接 import）。"""
    p = ROOT / "scripts" / filename
    spec = importlib.util.spec_from_file_location(module_name, p)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m

print("项目根目录：", ROOT)

项目根目录： G:\ThinkPad\Documents\jupyter_wenjian\商业数据分析\数据分析大作业


## 0. 一键跑完全部模型

In [2]:
an = load_step("step03", "03_analysis.py")
res = an.run_all()

[1/9] M1 同比环比 + 移动平均（48 个月）
  [√] 已保存结果表：t06_月度趋势_同比环比.csv
[2/9] M2 帕累托 / ABC 分类


  [√] 已保存结果表：t07_帕累托_ABC_子类.csv


  [√] 已保存结果表：t07_帕累托_ABC_客户.csv
[3/9] M3 RFM 八分类


  [√] 已保存结果表：t08a_RFM阈值.csv
  [√] 已保存结果表：t08_RFM客户明细.csv
  [√] 已保存结果表：t09_RFM八分类汇总.csv
      RFM 阈值：R=76 天，F=6 次，M=2,243 美元
[4/9] M4 K-Means 用户分群


  [√] 已保存结果表：t10a_KMeans轮廓系数.csv
  [√] 已保存结果表：t10_KMeans客群画像.csv
      K-Means 最优 K=3（轮廓系数 0.262）
[5/9] M5 同期群留存
  [√] 已保存结果表：t11_同期群留存矩阵.csv
  [√] 已保存结果表：t11b_平均留存曲线.csv
      同期群：12 个群，平均首月留存 11.7%
[6/9] M6 波士顿矩阵
  [√] 已保存结果表：t12_波士顿矩阵.csv
      BCG 分界线：份额 5.88%，整体增速 21.44%
[7/9] M7 时间序列预测（SARIMA vs Holt-Winters）


F:\anaconda3\envs\business_analysis\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:1022: EstimationWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  self._conditional_sum_squares(


... (17 outputs left out)

  [√] 已保存结果表：t14b_预测模型比选.csv


  [√] 已保存结果表：t13_预测结果.csv
  [√] 已保存结果表：t13b_预测回测明细.csv
  [√] 已保存结果表：t14_预测评估指标.csv
      最终模型：Holt-Winters（加性·阻尼），MAPE=16.93%，RMSE=19,886 美元
[8/9] M8 Apriori 关联规则


  [√] 已保存结果表：t15_关联规则.csv
      Apriori：频繁项集 115 个，有效规则 93 条（lift>1）
[9/9] M9 漏斗转化 + 折扣弹性


  [√] 已保存结果表：t16_履约漏斗.csv
      漏斗：5,111 → 2,281，端到端转化 44.6%
  [√] 已保存结果表：t17_折扣弹性.csv


## 1. M1 同比 / 环比 + 移动平均

**业务问题**：销售是在真增长，还是季节波动？
**口径**：同比 = 本月 / 去年同月 − 1；环比 = 本月 / 上月 − 1；MA3 = 近 3 个月均值。

In [3]:
res["trend"].tail(12)[["年月","销售额","同比","环比","销售额_MA3"]].round(3)

,年月,销售额,同比,环比,销售额_MA3
36,2026-01,44259.214,1.350,-0.546,73724.486
37,2026-02,20301.133,-0.117,-0.541,54020.875
38,2026-03,60728.481,0.145,1.991,41762.943
39,2026-04,36779.036,-0.053,-0.394,39269.550
40,2026-05,45155.482,-0.208,0.228,47554.333
41,2026-06,53056.078,0.296,0.175,44996.865
42,2026-07,45989.496,0.141,-0.133,48067.019
43,2026-08,64129.758,1.022,0.394,54391.777
44,2026-09,88064.532,0.198,0.373,66061.262
45,2026-10,83474.783,0.395,-0.052,78556.358


## 2. M2 帕累托 / ABC

**业务问题**：哪些少数子类撑起了大部分营收？

In [4]:
res["pareto_sub"].head(10)[["子类","销售额","销售额占比","累计占比","ABC等级"]]

,子类,销售额,销售额占比,累计占比,ABC等级
0,座椅,335768.2490,0.144321,0.144321,A 类
1,手机,331842.6400,0.142634,0.286955,A 类
2,收纳存储,224644.5540,0.096558,0.383513,A 类
3,桌子,208020.1820,0.089412,0.472925,A 类
4,装订用品,207354.8810,0.089126,0.562051,A 类
5,机器设备,189925.0310,0.081634,0.643685,A 类
6,配件,167380.3180,0.071944,0.715629,B 类
7,复印机,150745.2900,0.064794,0.780423,B 类
8,书架,115361.2043,0.049585,0.830008,B 类
9,小家电,108213.1850,0.046513,0.876521,B 类


In [5]:
import pandas as pd
print("A 类子类数：", (res["pareto_sub"]["ABC等级"]=="A 类").sum())
print("A 类销售额占比：", f'{res["pareto_sub"].loc[res["pareto_sub"]["ABC等级"]=="A 类","销售额占比"].sum():.1%}')
print("C 类子类数：", (res["pareto_sub"]["ABC等级"]=="C 类").sum())

A 类子类数： 6
A 类销售额占比： 64.4%
C 类子类数： 7


## 3. M3 RFM 八分类

**阈值依据**：R/F/M 三项均为右偏分布（少数大客户拉高均值），用中位数切分可保证高低两组各约 50%，
避免“高价值组只剩几十人”的失真。R 值越小越活跃，故 `R ≤ 中位数` 记为“高”。

In [6]:
pd.read_csv(ROOT/"outputs"/"tables"/"t08a_RFM阈值.csv")

,指标,取值
0,R 中位数(天),76.0
1,F 中位数(次),6.0
2,M 中位数(美元),2243.39
3,观察截止日,2026-12-31


In [7]:
res["rfm_sum"][["RFM客群","客户数","客户占比","销售额占比","平均R","平均F","平均M","毛利率"]]

,RFM客群,客户数,客户占比,销售额占比,平均R,平均F,平均M,毛利率
0,重要价值客户,204,0.2537,0.4024,33.3480,8.6422,4589.3653,0.1172
1,重要唤回客户,116,0.1443,0.2426,173.8017,8.1983,4864.7384,0.1592
2,重要深耕客户,31,0.0386,0.0627,35.6129,4.3548,4705.8517,0.1392
3,重要挽留客户,51,0.0634,0.0985,312.7843,4.5882,4493.8884,0.1186
4,一般保持客户,77,0.0958,0.0504,31.3506,7.5455,1522.0457,0.1118
5,一般发展客户,74,0.0920,0.0429,198.0946,7.1081,1349.4421,0.1379
6,潜力新客,93,0.1157,0.0380,34.3441,3.9677,951.3794,0.1072
7,流失客户,158,0.1965,0.0625,341.2278,3.5316,920.1895,0.0609


**读表要点**：`重要价值客户` 占客户数约 1/4 却贡献约 40% 销售额 —— 这是后续运营资源的投放依据。

## 4. M4 K-Means 用户分群

特征取 `log1p(R,F,M)` 后 Z-score 标准化（压缩右偏、统一量纲）；
K 在 3~6 之间按**轮廓系数**择优（下限 3 保证客群可运营，上限 6 避免切得太碎）。

In [8]:
pd.read_csv(ROOT/"outputs"/"tables"/"t10a_KMeans轮廓系数.csv")

,K值,轮廓系数,是否选中
0,3,0.2623,是
1,4,0.2592,NaN
2,5,0.2515,NaN
3,6,0.2423,NaN


In [9]:
res["km_prof"][["业务标签","客户数","客户占比","销售额占比","平均R","平均F","平均M","毛利率"]]

,业务标签,客户数,客户占比,销售额占比,平均R,平均F,平均M,毛利率
0,核心高价值群,301,0.3744,0.4877,29.1894,8.0631,3769.2404,0.1160
1,沉睡高额群,350,0.4353,0.4738,188.5657,6.3171,3149.5135,0.1166
2,低频休眠群,153,0.1903,0.0385,283.7712,3.1307,586.0345,0.0819


## 5. M5 同期群留存

以客户**首次购买月份**为同期群，统计其后第 n 个月仍下单的客户占比。
只保留群规模 ≥20 的同期群，避免小样本造成的 0%/100% 假象。

In [10]:
co = res["cohort"]
co.iloc[:, :8]

,首购月,群规模,第0期,第1期,第2期,第3期,第4期,第5期
0,2023-01,33,1.0,0.090909,NaN,0.060606,0.060606,NaN
1,2023-02,24,1.0,0.166667,0.083333,0.041667,NaN,0.083333
2,2023-03,66,1.0,0.075758,0.030303,0.121212,0.106061,NaN
3,2023-04,55,1.0,0.109091,0.036364,0.072727,0.054545,0.127273
4,2023-05,57,1.0,0.087719,0.087719,0.070175,0.192982,0.035088
5,2023-06,49,1.0,0.020408,0.040816,0.102041,0.040816,0.183673
6,2023-07,44,1.0,0.136364,0.113636,NaN,0.113636,0.181818
7,2023-08,52,1.0,0.173077,0.057692,0.211538,0.096154,0.057692
8,2023-09,68,1.0,0.132353,0.132353,0.176471,0.014706,0.029412
9,2023-10,43,1.0,0.069767,0.116279,0.023256,0.069767,0.093023


## 6. M6 波士顿矩阵

横轴=销售额份额，纵轴=同比增速，气泡=销售额，颜色=毛利率。
**分界线**：横轴取平均份额（1/17），纵轴取整体增速，保证四象限与业务口径一致。

In [11]:
res["bcg"][["子类","2025销售额","2026销售额","同比增速","销售额份额","毛利率","象限"]]

,子类,2025销售额,2026销售额,同比增速,销售额份额,毛利率,象限
0,手机,79178.0100,105668.3920,0.334567,0.141729,0.122576,明星业务
1,座椅,85079.4310,98032.3770,0.152245,0.131487,0.079444,现金牛业务
2,装订用品,51580.1330,73651.2690,0.427900,0.098786,0.107817,明星业务
3,收纳存储,58845.6760,70273.8100,0.194205,0.094255,0.104782,现金牛业务
4,复印机,49599.4100,62916.6680,0.268496,0.084388,0.397985,明星业务
5,桌子,60835.4105,60999.2025,0.002692,0.081816,-0.132662,现金牛业务
6,配件,41895.8540,59946.2320,0.430839,0.080403,0.261440,明星业务
7,机器设备,55906.8860,43887.8750,-0.214983,0.058865,-0.064496,现金牛业务
8,小家电,26164.2350,43111.1560,0.647713,0.057823,0.183595,问题业务
9,书架,26426.8805,30354.0737,0.148606,0.040713,-0.021684,瘦狗业务


In [12]:
print("象限分布：")
print(res["bcg"].groupby("象限").agg(子类数=("子类","count"), 销售额=("2026销售额","sum"),
                                毛利率均值=("毛利率","mean")).round(3))

象限分布：
       子类数         销售额  毛利率均值
象限                           
明星业务     4  302182.561  0.222
现金牛业务    4  273193.264 -0.003
瘦狗业务     4   79735.584  0.123
问题业务     5   90456.122  0.320


## 7. M7 时间序列预测（SARIMA vs Holt-Winters）

流程：
1. 前 42 个月训练、末尾 6 个月回测；
2. SARIMA 在 p,q∈{0,1,2}、P,Q∈{0,1}、d=D=1、s=12 的网格中按 AIC 择优；
3. 同时拟合两种 Holt-Winters（阻尼/非阻尼）；
4. **以样本外 MAPE 最小者为最终模型**（预测任务更应看样本外误差，AIC 只在同一模型族内选阶）；
5. 输出未来 6 个月点预测 + 95% 置信区间。

In [13]:
res["sarima"]["compare"]

,模型,回测MAPE,回测RMSE,回测MAE,AIC,是否最终模型
0,Holt-Winters（加性·阻尼）,0.169285,19886.115890,14641.511443,NaN,是
1,Holt-Winters（加性）,0.184883,20099.824254,15735.647620,NaN,
2,"SARIMA(2, 1, 2)x(0, 1, 1, 12)",0.253590,25031.501765,20909.420824,309.574312,


In [14]:
pd.read_csv(ROOT/"outputs"/"tables"/"t14_预测评估指标.csv")

,指标,取值
0,最终模型,Holt-Winters（加性·阻尼）
1,SARIMA 阶数（AIC 最优）,"SARIMA(2, 1, 2)x(0, 1, 1, 12)"
2,回测期数,6 个月（2026-07 至 2026-12）
3,MAPE（回测）,16.93%
4,RMSE（回测，美元）,"19,886"
5,MAE（回测，美元）,"14,642"
6,预测期数,6 个月
7,置信水平,95%


In [15]:
res["sarima"]["forecast"]

,年月,预测销售额,下限95,上限95
0,2027-01,50876.88,28390.00,74153.04
1,2027-02,42776.39,19606.62,66838.93
2,2027-03,77559.02,53817.70,104249.51
3,2027-04,62719.81,35432.17,91147.44
4,2027-05,69946.51,37367.54,103675.28
5,2027-06,66790.91,26745.07,106711.71


## 8. M8 Apriori 关联规则

订单=购物篮，子类=商品（17 类，避免 SKU 过细导致支持度过低）。
支持度 0.6%（约 30 单）、置信度 15%、仅保留 lift>1 的有效关联。

In [16]:
res["rules"].head(10)[["前项","后项","支持度","置信度","提升度"]]

,前项,后项,支持度,置信度,提升度
0,家居陈设、装订用品,小家电,0.006652,0.170000,1.892963
1,手机、收纳存储,家居陈设,0.007435,0.324786,1.806293
2,纸张、装订用品,小家电,0.008609,0.159420,1.775157
3,美术用品、装订用品,座椅,0.006065,0.194969,1.686099
4,家居陈设、收纳存储,手机,0.007435,0.271429,1.679505
5,座椅、装订用品,美术用品,0.006065,0.246032,1.663318
6,家居陈设、纸张,收纳存储,0.009000,0.254144,1.650481
7,美术用品、装订用品,手机,0.008218,0.264151,1.634474
8,装订用品、配件,收纳存储,0.007826,0.248447,1.613486
9,家居陈设、手机,收纳存储,0.007435,0.246753,1.602485


## 9. M9 履约漏斗 + 折扣弹性（商业建议的数据支撑）

漏斗以**订单**为粒度，逐级要求满足前序条件，体现真实流失。

In [17]:
res["funnel"]

,环节,订单数,整体转化率,环节转化率
0,下单订单,5111,1.000000,1.000000
1,按时履约（≤4 天）,3078,0.602230,0.602230
2,未被退货,2893,0.566034,0.939896
3,订单盈利,2281,0.446292,0.788455


In [18]:
res["discount"][["折扣档","销售额占比","毛利率","亏损占比"]]

,折扣档,销售额占比,毛利率,亏损占比
0,无折扣,0.475095,0.295586,0.000000
1,0~10%,0.023620,0.165597,0.041667
2,10~20%,0.344503,0.115408,0.140234
3,20~30%,0.044905,-0.100632,0.917391
4,30~40%,0.056303,-0.194498,0.888889
5,40~50%,0.027682,-0.357116,1.000000
6,50%以上,0.027892,-1.187064,1.000000


**关键结论**：折扣超过 20% 后毛利率迅速转负、亏损明细占比飙升 —— 这是第 6 章“折扣红线”建议的直接依据。

## 10. 结果表清单

所有模型结果已落盘到 `outputs/tables/`，图表与论文中的每个数字都能回溯到具体表格。

In [19]:
import os
for f in sorted(os.listdir(ROOT/"outputs"/"tables")):
    print(f)

t01_数据体检报告.csv
t02_数据规模指标.csv
t03_字段字典.csv
t04_清洗日志.csv
t05_数据质量总览.csv
t06_月度趋势_同比环比.csv
t07_帕累托_ABC_子类.csv
t07_帕累托_ABC_客户.csv
t08_RFM客户明细.csv
t08a_RFM阈值.csv
t09_RFM八分类汇总.csv
t10_KMeans客群画像.csv
t10a_KMeans轮廓系数.csv
t11_同期群留存矩阵.csv
t11b_平均留存曲线.csv
t12_波士顿矩阵.csv
t13_SARIMA预测.csv
t13_预测结果.csv
t13b_SARIMA回测.csv
t13b_预测回测明细.csv
t14_SARIMA评估指标.csv
t14_预测评估指标.csv
t14b_预测模型比选.csv
t15_关联规则.csv
t15b_子类共现矩阵.csv
t16_履约漏斗.csv
t17_折扣弹性.csv
